# Phase 2 — Phân tích Semantic ID

Notebook này chỉ đọc `semantic_ids.parquet` do notebook 03 tạo ra. Không load checkpoint, không chạy RQ-VAE và không cần GPU.

Mục tiêu là kiểm tra raw SID ba tầng có tạo ra cluster hợp lý hay gần như biến mỗi sản phẩm thành một SID riêng.

## 0. Cấu hình

In [ ]:
from pathlib import Path

SEMANTIC_ID_ROOT = None
CODEBOOK_SIZE = 256
OUTPUT_ROOT = (
    Path("/kaggle/working/semantic_id_analysis")
    if Path("/kaggle/working").exists()
    else Path.cwd().parent / "output/semantic-id-analysis"
)

print("Configuration loaded.")

## 1. Tìm và đọc Semantic ID

In [ ]:
import json

import numpy as np
import pandas as pd


def locate_semantic_id_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if (root / "semantic_ids.parquet").is_file():
            return root
        raise FileNotFoundError(f"semantic_ids.parquet was not found in {root}")

    candidates = [
        Path.cwd() / "ai-recommendation/output/rq-vae",
        Path.cwd() / "output/rq-vae",
        Path.cwd().parent / "output/rq-vae",
        Path("/kaggle/working/rq-vae"),
        Path("/kaggle/working/vmarket_rqvae"),
    ]
    for candidate in candidates:
        if (candidate / "semantic_ids.parquet").is_file():
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = list(kaggle_input.glob("**/output/rq-vae/semantic_ids.parquet"))
        if not matches:
            matches = list(kaggle_input.glob("**/semantic_ids.parquet"))
        if matches:
            return matches[0].parent.resolve()

    raise FileNotFoundError(
        "semantic_ids.parquet was not found. Add notebook 03 output as Kaggle Data or set SEMANTIC_ID_ROOT."
    )


SEMANTIC_ID_ROOT = locate_semantic_id_root(SEMANTIC_ID_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
semantic_ids = pd.read_parquet(SEMANTIC_ID_ROOT / "semantic_ids.parquet")
sid_columns = [column for column in semantic_ids.columns if column.startswith("sid_")]
sid_columns.sort(key=lambda column: int(column.split("_")[1]))

expected_columns = ["product_index", "product_id", *sid_columns]
if list(semantic_ids.columns) != expected_columns:
    raise ValueError(f"Unexpected columns: {list(semantic_ids.columns)}")
if not sid_columns:
    raise ValueError("No SID columns were found.")
if semantic_ids["product_id"].duplicated().any():
    raise ValueError("product_id must be unique.")

print("SEMANTIC_ID_ROOT:", SEMANTIC_ID_ROOT)
print("Products:", f"{len(semantic_ids):,}")
print("SID columns:", sid_columns)
display(semantic_ids.head())

## 2. Phân bố cluster hoàn chỉnh

Mỗi bộ `(sid_0, sid_1, sid_2)` là một cluster. Các chỉ số quan trọng nhất là tỷ lệ sản phẩm singleton và kích thước cluster theo góc nhìn của sản phẩm.

In [ ]:
cluster_sizes = (
    semantic_ids.groupby(sid_columns, sort=False, observed=True)
    .size()
    .rename("cluster_size")
    .reset_index()
)
sizes = cluster_sizes["cluster_size"].to_numpy()
product_count = len(semantic_ids)
cluster_count = len(cluster_sizes)
probabilities = sizes / product_count
entropy = float(-(probabilities * np.log(probabilities)).sum())


def weighted_quantile(values, weights, quantile):
    order = np.argsort(values)
    sorted_values = values[order]
    cumulative = np.cumsum(weights[order])
    position = np.searchsorted(cumulative, quantile * cumulative[-1], side="left")
    return int(sorted_values[min(position, len(sorted_values) - 1)])


metrics = {
    "product_count": product_count,
    "occupied_sid_count": cluster_count,
    "theoretical_sid_capacity": CODEBOOK_SIZE ** len(sid_columns),
    "sid_entropy_nats": entropy,
    "effective_sid_count": float(np.exp(entropy)),
    "average_items_per_sid": float(product_count / cluster_count),
    "singleton_cluster_count": int((sizes == 1).sum()),
    "singleton_cluster_rate": float((sizes == 1).mean()),
    "singleton_item_rate": float((sizes == 1).sum() / product_count),
    "items_in_shared_clusters_rate": float(sizes[sizes > 1].sum() / product_count),
    "cluster_size_median": float(np.median(sizes)),
    "cluster_size_p90": float(np.quantile(sizes, 0.90)),
    "cluster_size_p95": float(np.quantile(sizes, 0.95)),
    "cluster_size_p99": float(np.quantile(sizes, 0.99)),
    "cluster_size_max": int(sizes.max()),
    "item_weighted_cluster_size_median": weighted_quantile(sizes, sizes, 0.50),
    "item_weighted_cluster_size_p90": weighted_quantile(sizes, sizes, 0.90),
    "item_weighted_cluster_size_p95": weighted_quantile(sizes, sizes, 0.95),
    "item_weighted_cluster_size_p99": weighted_quantile(sizes, sizes, 0.99),
}

display(pd.Series(metrics, name="value").to_frame())

## 3. Mức sử dụng codebook và prefix

Bảng prefix cho biết candidate set thu hẹp như thế nào sau từng tầng SID.

In [ ]:
codebook_rows = []
for column in sid_columns:
    used = int(semantic_ids[column].nunique())
    codebook_rows.append({
        "layer": column,
        "used_codes": used,
        "codebook_size": CODEBOOK_SIZE,
        "usage_rate": used / CODEBOOK_SIZE,
    })
codebook_usage = pd.DataFrame(codebook_rows)

prefix_rows = []
for depth in range(1, len(sid_columns) + 1):
    prefix = sid_columns[:depth]
    prefix_sizes = semantic_ids.groupby(prefix, sort=False, observed=True).size().to_numpy()
    prefix_rows.append({
        "prefix_depth": depth,
        "occupied_prefixes": len(prefix_sizes),
        "mean_items": prefix_sizes.mean(),
        "median_items": np.median(prefix_sizes),
        "p95_items": np.quantile(prefix_sizes, 0.95),
        "p99_items": np.quantile(prefix_sizes, 0.99),
        "max_items": prefix_sizes.max(),
    })
prefix_summary = pd.DataFrame(prefix_rows)

display(codebook_usage)
display(prefix_summary)

## 4. Biểu đồ phân bố

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

if sizes.max() > 1:
    bins = np.logspace(0, np.log10(sizes.max() + 1), 40)
    axes[0].hist(sizes, bins=bins)
    axes[0].set_xscale("log")
    axes[0].set_yscale("log")
else:
    axes[0].bar([1], [len(sizes)])
axes[0].set_title("Cluster size distribution")
axes[0].set_xlabel("Items per SID")
axes[0].set_ylabel("Number of SIDs")

axes[1].bar(codebook_usage["layer"], codebook_usage["usage_rate"])
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Codebook usage")
axes[1].set_ylabel("Usage rate")

plt.tight_layout()
plt.show()

## 5. Các cluster lớn nhất

Hiển thị tối đa năm `product_id` mẫu trong mỗi cluster lớn để hỗ trợ kiểm tra thủ công.

In [ ]:
largest_clusters = cluster_sizes.nlargest(20, "cluster_size")
largest_members = semantic_ids.merge(largest_clusters[sid_columns], on=sid_columns, how="inner")
sample_products = (
    largest_members.groupby(sid_columns, sort=False)["product_id"]
    .agg(lambda values: list(values.head(5)))
    .rename("sample_product_ids")
    .reset_index()
)
largest_clusters = largest_clusters.merge(sample_products, on=sid_columns, how="left")
display(largest_clusters)

## 6. Lưu kết quả phân tích

In [ ]:
cluster_sizes.to_parquet(OUTPUT_ROOT / "semantic_id_cluster_sizes.parquet", index=False)
codebook_usage.to_csv(OUTPUT_ROOT / "codebook_usage.csv", index=False)
prefix_summary.to_csv(OUTPUT_ROOT / "prefix_summary.csv", index=False)
with (OUTPUT_ROOT / "semantic_id_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print("Semantic ID analysis: PASSED")
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## Cách đọc kết quả

- `singleton_item_rate` cao nghĩa là phần lớn sản phẩm không thực sự chia sẻ cluster với sản phẩm khác.
- `item_weighted_cluster_size_*` mô tả kích thước cluster mà một sản phẩm điển hình rơi vào; nó hữu ích hơn median tính trên cluster khi có nhiều singleton.
- `prefix_summary` cho biết số candidate còn lại sau khi Transformer sinh từng token SID.
- Chưa nên kết luận chỉ từ reconstruction loss; quyết định giữ `256 × 256 × 256` hay giảm codebook phải dựa vào các số này.